# 01. 원본 데이터 개요

`data/raw`의 Olist CSV 9개를 읽어 테이블별로 아래 항목을 확인한다.

- 행 수, 열 수
- 컬럼명과 데이터 타입
- 컬럼별 결측치 개수와 비율
- 중복 행 개수
- 상위 3행 미리보기

원본 파일은 읽기만 하고 수정하지 않는다.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


def find_project_root(start: Path) -> Path:
    """data/raw 폴더가 있는 프로젝트 루트를 찾는다."""
    for path in [start, *start.parents]:
        if (path / "data" / "raw").is_dir():
            return path
    raise FileNotFoundError("프로젝트 루트(data/raw)를 찾을 수 없습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
csv_files = sorted(RAW_DIR.glob("*.csv"))
print(f"CSV 파일 {len(csv_files)}개")
for f in csv_files:
    print(f" - {f.name}")

CSV 파일 9개
 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_orders_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv


## CSV 읽기

기본은 `encoding="utf-8"`로 읽고, 첫 번째 컬럼명 앞에 BOM(`\ufeff`) 같은 이상한 문자가 붙어 있으면 그 파일만 `encoding="utf-8-sig"`로 다시 읽는다.

In [2]:
def has_strange_prefix(column_name: str) -> bool:
    """컬럼명 앞에 BOM 등 이상한 문자가 붙어 있는지 확인한다."""
    first_char = str(column_name)[:1]
    return first_char == "\ufeff" or not (first_char.isalnum() or first_char == "_")


def read_csv_safely(path: Path) -> tuple[pd.DataFrame, str]:
    """utf-8로 읽고, 첫 컬럼명이 이상하면 utf-8-sig로 다시 읽는다."""
    df = pd.read_csv(path, encoding="utf-8")
    if has_strange_prefix(df.columns[0]):
        print(f"[{path.name}] 첫 컬럼명 {df.columns[0]!r} → utf-8-sig로 다시 읽음")
        return pd.read_csv(path, encoding="utf-8-sig"), "utf-8-sig"
    return df, "utf-8"


tables: dict[str, pd.DataFrame] = {}
encodings: dict[str, str] = {}
for path in csv_files:
    name = path.stem
    tables[name], encodings[name] = read_csv_safely(path)

print(f"\n{len(tables)}개 테이블 로드 완료")


9개 테이블 로드 완료


## 테이블별 상세 확인

In [3]:
def column_profile(df: pd.DataFrame) -> pd.DataFrame:
    """컬럼별 데이터 타입, 결측치 개수와 비율을 표로 만든다."""
    null_count = df.isna().sum()
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_count": null_count,
        "null_ratio(%)": (null_count / len(df) * 100).round(2),
    })


def describe_table(name: str, df: pd.DataFrame) -> None:
    """테이블 하나의 기본 정보를 출력한다."""
    print("=" * 80)
    print(f"[{name}]  (encoding: {encodings[name]})")
    print(f"행 수: {df.shape[0]:,} / 열 수: {df.shape[1]}")
    print(f"중복 행 개수: {df.duplicated().sum():,}")
    print("\n컬럼 정보 (타입, 결측치)")
    display(column_profile(df))
    print("상위 3행")
    display(df.head(3))


for name, df in tables.items():
    describe_table(name, df)

[olist_customers_dataset]  (encoding: utf-8)
행 수: 99,441 / 열 수: 5
중복 행 개수: 0

컬럼 정보 (타입, 결측치)


,dtype,null_count,null_ratio(%)
customer_id,str,0,0.0
customer_unique_id,str,0,0.0
customer_zip_code_prefix,int64,0,0.0
customer_city,str,0,0.0
customer_state,str,0,0.0


상위 3행


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


[olist_geolocation_dataset]  (encoding: utf-8)
행 수: 1,000,163 / 열 수: 5


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


[product_category_name_translation]  (encoding: utf-8)
행 수: 71 / 열 수: 2
중복 행 개수: 0

컬럼 정보 (타입, 결측치)


,dtype,null_count,null_ratio(%)
product_category_name,str,0,0.0
product_category_name_english,str,0,0.0


상위 3행


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## 9개 테이블 요약

In [4]:
summary = pd.DataFrame([
    {
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "cols_with_nulls": int((df.isna().sum() > 0).sum()),
    }
    for name, df in tables.items()
]).set_index("table")

summary

,rows,columns,cols_with_nulls
table,,,
olist_customers_dataset,99441,5,0
olist_geolocation_dataset,1000163,5,0
olist_order_items_dataset,112650,7,0
olist_order_payments_dataset,103886,5,0
olist_order_reviews_dataset,99224,7,2
olist_orders_dataset,99441,8,3
olist_products_dataset,32951,9,8
olist_sellers_dataset,3095,4,0
product_category_name_translation,71,2,0
